In [1]:
import numpy as np
import cv2
import base64

def decode_base64(b64_string):
    """Converts Flutter Base64 to OpenCV RGB image"""
    if "," in b64_string:
        b64_string = b64_string.split(",")[1]
    img_data = base64.b64decode(b64_string)
    nparr = np.frombuffer(img_data, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def preprocess_image(img, target_size=(112, 112)):
    """Resizes and normalizes image to [-1, 1]"""
    if img.shape[0:2] != target_size:
        img = cv2.resize(img, target_size)
    img = (img.astype(np.float32) - 127.5) / 127.5
    return np.expand_dims(img, axis=0)

def generate_embedding(model, b64_image):
    """Turns a Base64 string into a normalized 128-float list"""
    img = decode_base64(b64_image)
    tensor = preprocess_image(img)
    embedding = model.predict(tensor, verbose=0)[0]
    
    # L2 Normalization
    norm = np.linalg.norm(embedding)
    normalized_vec = embedding / (norm + 1e-7)
    return normalized_vec.tolist()